<a href="https://colab.research.google.com/github/jiyeonlee-2930/HandsOn-LLM/blob/main/Transform_Encoder_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python transformer_encoder_practice.py

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=145)

TOKENS = ['<sos>', '오늘', '학교', '도서관', '운영', '시간을', '알려', '주세요', '?', '<eos>', '<pad>']
#토큰 임베딩
E = np.array([
    [.20,-.10,.40,.30,-.20,.10],
    [.30,-.80,.50,.20,-.40,.60],
    [-.20,.70,.10,-.50,.40,.30],
    [.80,.20,-.60,.50,.10,-.30],
    [.40,-.30,.70,.10,-.50,.20],
    [-.10,.60,.30,-.20,.80,.40],
    [.50,.10,-.40,.70,.20,-.60],
    [.20,.50,.60,-.30,.10,.40],
    [-.40,.20,.10,.30,-.10,.50],
    [.10,-.20,.30,.40,.20,-.10],
    [0,0,0,0,0,0]
], dtype=float)
# 위치 임베딩
P = np.array([
    [0,1,0,1,0,1],
    [.8415,.5403,.0464,.9989,.0022,1],
    [.9093,-.4161,.0927,.9957,.0043,1],
    [.1411,-.9900,.1388,.9903,.0065,1],
    [-.7568,-.6536,.1846,.9828,.0086,1],
    [-.9589,.2837,.2300,.9732,.0108,.9999],
    [-.2794,.9602,.2749,.9615,.0129,.9999],
    [.6570,.7539,.3192,.9477,.0151,.9999],
    [.9894,-.1455,.3629,.9318,.0172,.9999],
    [.4121,-.9111,.4057,.9140,.0194,.9998],
    [-.5440,-.8391,.4477,.8942,.0215,.9998]
], dtype=float)
#가중치 행렬
WQ = np.array([[.2,.1,0],[0,.3,.1],[.1,0,.2],[.3,.1,.3],[0,.2,.3],[.1,0,.2]])
WK = np.array([[.1,.2,0],[.2,0,.1],[0,.1,.3],[.1,.3,0],[.2,0,.1],[0,.1,.2]])
WV = np.array([[.3,0,.1],[.1,.2,0],[0,.3,.1],[.2,0,.2],[.1,.1,.3],[0,.2,.1]])

# PDF does not supply W_O or FFN weights: the following are illustrative, NOT worksheet answers.
WO = np.array([[.2,0,.1,.1,0,.2],[0,.2,.1,0,.2,.1],[.1,.1,0,.2,.1,0]])
rng = np.random.default_rng(42)
W1 = rng.normal(0, .15, (6, 12))
b1 = np.zeros(12)
W2 = rng.normal(0, .15, (12, 6))
b2 = np.zeros(6)

In [ ]:
def softmax(x, axis=-1):
    z = x - np.max(x, axis=axis, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=axis, keepdims=True)

def layer_norm(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = ((x-mu)**2).mean(axis=-1, keepdims=True)
    return (x-mu) / np.sqrt(var+eps)  # gamma=1, beta=0 (illustrative)

def show(title, x):
    print('\n' + '='*75)
    print(title)
    print(x)

In [ ]:
show('1. 토큰과 ID', list(enumerate(TOKENS)))
X = E + P
# The worksheet includes positional embedding for <pad>; for actual attention, exclude pad as a key.
show('2. 입력 임베딩 X = E + P (11 x 6)', X)
i = TOKENS.index('오늘')
show('2. 오늘 입력 벡터 (1 x 6)', X[i])
Q, K, V = X @ WQ, X @ WK, X @ WV
show('3. 오늘 Q = X[오늘] @ WQ (3차원)', Q[i])
show('3. 오늘 K = X[오늘] @ WK (3차원)', K[i])
show('3. 오늘 V = X[오늘] @ WV (3차원)', V[i])
show('4. Q,K,V의 모양', {'X': X.shape, 'Q': Q.shape, 'K': K.shape, 'V': V.shape})
raw_scores = Q @ K.T
scaled_scores = raw_scores / np.sqrt(3)
# <pad> cannot be attended to as a key.
scaled_scores[:, -1] = -np.inf
show('4. Scaled QK^T / sqrt(3) - 오늘 행 (pad 위치 -inf)', scaled_scores[i])
A = softmax(scaled_scores, axis=-1)
show('5. softmax 후 오늘 Attention 가중치 (합계=1)', A[i])
show('5. 가중치 합계', A[i].sum())
context = A @ V
show('6. Attention @ V: 전체 Context (11 x 3)', context)
show('6. 오늘 Context (3차원)', context[i])
projected = context @ WO
show('7. 실습용 WO로 3차원 -> 6차원 투영 (오늘)', projected[i])
residual1 = X + projected
show('7. Residual: X + AttentionOutput (오늘)', residual1[i])
norm1 = layer_norm(residual1)
show('8. LayerNorm 1 (오늘)', norm1[i])
ffn = np.maximum(0, norm1 @ W1 + b1) @ W2 + b2
show('9. ReLU 기반 FFN 출력 (오늘)', ffn[i])
final = layer_norm(norm1 + ffn)
show('10. Add & LayerNorm 후 Encoder 최종 출력 (오늘)', final[i])
show('10. 최종 Encoder 출력 모양', final.shape)
print('\n주의: 7~10번의 수치는 PDF에 없는 WO, W1, W2와 LayerNorm 기본값을 사용한 실습 예시입니다.')
print('주의: 이 코드는 Encoder 순전파 계산이며 BERT 사전학습/미세튜닝은 수행하지 않습니다.')